# Multimodal RAG 智能问答系统 Demo

基于 `jina-embeddings-v5-omni-small` 统一多模态嵌入空间。

本 Notebook 演示:
1. 预处理管线 (文本/图片/音频/视频)
2. 向量存储与跨模态检索
3. 完整 RAG 查询流程 (需要 API key)
4. 跨模态搜索展示

## 环境准备
```bash
pip install -r requirements.txt
cp .env.example .env  # 编辑填入 API keys
```

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import numpy as np

# 核心组件
from src.embeddings.jina_embedder import JinaEmbedder, Modality, EmbeddingInput
from src.preprocessing.text_chunker import TextChunker
from src.preprocessing.image_processor import ImageProcessor
from src.storage.vector_store import VectorStore
from src.storage.schemas import MediaChunk
from src.retrieval.retriever import MultiModalRetriever

print('✅ 导入成功')

## 1. 文本分块演示

In [ ]:
chunker = TextChunker(chunk_size=256, chunk_overlap=32)

# 读取示例文本
sample_path = Path('../data/text/sample_article.txt')
if sample_path.exists():
    text = sample_path.read_text()
    chunks = chunker.chunk_with_token_limit(text, max_tokens=256, source_file=str(sample_path))
    print(f'原文长度: {len(text)} 字符')
    print(f'分为 {len(chunks)} 个 chunks:\n')
    for c in chunks:
        print(f'--- Chunk {c["metadata"]["chunk_index"]} ({c["metadata"]["token_count"]} tokens) ---')
        print(c['text'][:150] + '...')
        print()

## 2. 图片处理演示

In [ ]:
from IPython.display import Image as IPImage, display
import base64
from io import BytesIO
from PIL import Image

img_proc = ImageProcessor(max_dim=512, thumbnail_dim=128)

img_path = Path('../data/images/sunset_ocean.jpg')
if img_path.exists():
    result = img_proc.process(str(img_path))
    
    print(f'原始尺寸: {result["metadata"]["original_size"]}')
    print(f'处理后尺寸: {result["metadata"]["resized_size"]}')
    print(f'JPEG 大小: {len(result["image_bytes"])} bytes')
    print(f'Base64 长度: {len(result["base64"])} chars')
    print()
    
    # 显示原始图
    print('原始图片:')
    display(IPImage(str(img_path)))
    
    # 显示缩略图
    print('缩略图:')
    thumb_bytes = base64.b64decode(result['thumbnail_base64'])
    display(IPImage(data=thumb_bytes))
else:
    print(f'图片不存在: {img_path}')

## 3. 向量存储与检索演示 (使用模拟数据)

In [ ]:
import tempfile

# 创建临时向量库
tmpdir = tempfile.mkdtemp()
store = VectorStore(
    collection_name='notebook_demo',
    persist_directory=tmpdir,
    embedding_dim=64,
)

# 模拟多模态数据
np.random.seed(42)
n = 20
dummy_embeddings = np.random.randn(n, 64).astype(np.float32)

dummy_chunks = []
for i in range(n):
    if i < 8:
        mod = Modality.TEXT
        preview = f'文本片段 {i}: 关于人工智能和多模态学习的内容...'
    elif i < 14:
        mod = Modality.IMAGE
        preview = f'[图片 {i}: 日落海岸照片]'
    else:
        mod = Modality.AUDIO
        preview = f'[音频 {i}: 00:{i*3}-00:{(i+1)*3}]'
    
    dummy_chunks.append(MediaChunk(
        chunk_id=f'nb_demo_{i}',
        modality=mod,
        content_type=f'{mod.value}_chunk',
        source_file=f'demo/sample_{i}',
        source_file_name=f'sample_{i}',
        content_preview=preview,
        text_content=preview if mod == Modality.TEXT else None,
        embedding_dim=64,
    ))

store.add(dummy_embeddings, dummy_chunks)
print(f'已添加 {store.count()} 个 chunks')

# 跨模态搜索
query_vec = np.random.randn(64).astype(np.float32)
cross_results = store.search_cross_modal(query_vec, top_k_per_modality=3)

for mod, results in cross_results.items():
    print(f'\n{mod.upper()} ({len(results)} 条):')
    for r in results:
        print(f'  [{r.score:.3f}] {r.chunk.content_preview[:60]}')

# 清理
store.clear()

## 4. 完整 RAG 查询 (需要 API Keys)

⚠️ 以下代码需要配置 `JINA_API_KEY` 和 `ANTHROPIC_API_KEY`。

In [ ]:
# 完整 RAG 查询示例 (需要 API keys)
"""
from src.pipeline.ingestion import IngestionPipeline
from src.pipeline.query import QueryPipeline

# 1. 入库数据
ingestion = IngestionPipeline()
stats = ingestion.ingest_directory('../data/')
print(f'入库完成: {stats}')

# 2. 查询
query = QueryPipeline()

# 普通查询
response = query.query('多模态学习有什么应用?')
print(f'回答: {response.answer}')
print(f'延迟: {response.latency_breakdown}')

# 跨模态查询 (文本查询同时检索图片)
response = query.cross_modal_query('sunset')
print(f'回答: {response.answer}')
print(f'模态分布: {response.modality_breakdown}')

# 仅检索不生成 (查看检索质量)
result = query.retrieve_only('海洋', top_k=5)
for r in result['results']:
    print(f'[{r["modality"]}] {r["content_preview"][:50]}... ({r["score"]:.3f})')
"""

print('取消注释以上代码以运行完整 RAG 查询')
print('请先在 ../.env 中配置 API keys')

## 5. 系统架构图

```
┌─────────────────────────────────────────────────────────────────┐
│                  Multimodal RAG Pipeline                         │
│                                                                  │
│  [Ingestion Pipeline]              [Query Pipeline]              │
│  ┌──────────┐  ┌───────────────┐   ┌──────────────┐            │
│  │ 文件上传  │→│ 模态检测路由   │   │  用户查询文本  │            │
│  │.txt .jpg │ │               │   └──────┬───────┘            │
│  │.mp3 .mp4 │ │ TEXT→TextChunker│         │                    │
│  └──────────┘ │ IMG→ImgProc    │   ┌──────▼───────┐            │
│               │ AUD→AudProc    │   │ JinaEmbedder  │            │
│               │ VID→VidProc    │   │ embed_query() │            │
│               └───────┬───────┘   └──────┬───────┘            │
│                       │                  │                     │
│               ┌───────▼───────┐   ┌──────▼───────┐            │
│               │ JinaEmbedder   │   │ MultiModal   │            │
│               │ embed_batch() │   │ Retriever     │            │
│               └───────┬───────┘   └──────┬───────┘            │
│                       │                  │                     │
│               ┌───────▼──────────────────▼───────┐            │
│               │        ChromaDB Vector Store      │            │
│               │   (统一嵌入空间, 元数据过滤)        │            │
│               └──────────────────┬───────────────┘            │
│                                  │                             │
│                          ┌───────▼───────┐                    │
│                          │  PromptTemplate│                    │
│                          │  上下文组装     │                    │
│                          └───────┬───────┘                    │
│                                  │                             │
│                          ┌───────▼───────┐                    │
│                          │   LLM Backend  │                    │
│                          │ Claude/OpenAI  │                    │
│                          └───────┬───────┘                    │
│                                  │                             │
│                          ┌───────▼───────┐                    │
│                          │  QueryResponse │                    │
│                          │ answer+sources │                    │
│                          └───────────────┘                    │
└─────────────────────────────────────────────────────────────────┘
```

## 6. 跨模态嵌入空间可视化

jina-embeddings-v5-omni-small 将不同模态映射到同一向量空间,
使得「日落」的文本描述与日落图片的嵌入向量在空间中接近。

这使得:
- 文字查询 "sunset" 可以检索到日落图片
- 图片查询可以检索到相关的文本描述
- 音频和视频片段也能被语义相关的文字检索到